<a href="https://colab.research.google.com/github/wvb20/cv-face-alignment/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === Mount Google Drive and create persistent project folders ===
import os
from google.colab import drive

drive.mount('/content/drive')

# Persistent workspace on Drive — survives Colab session restarts
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/cv-face-alignment'
os.makedirs(f'{DRIVE_PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/figures', exist_ok=True)

print(f'✓ Drive mounted')
print(f'✓ Project workspace: {DRIVE_PROJECT_DIR}')
!ls -la {DRIVE_PROJECT_DIR}

ModuleNotFoundError: No module named 'google'

In [ ]:
# === Confirm runtime environment ===
import sys, torch, cv2, numpy as np, sklearn

print(f'Python:       {sys.version.split()[0]}')
print(f'NumPy:        {np.__version__}')
print(f'OpenCV:       {cv2.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA avail:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')

Python:       3.12.13
NumPy:        2.0.2
OpenCV:       4.13.0
scikit-learn: 1.6.1
PyTorch:      2.10.0+cu128
CUDA avail:   True
GPU:          Tesla T4


In [ ]:
# === Cell 3: Clone repo into Colab VM and add src/ to path ===
import sys

GITHUB_USERNAME = 'wvb20'
REPO_NAME       = 'cv-face-alignment'
REPO_PATH       = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

# Add the project ROOT to sys.path so `from src import ...` works.
# (Adding REPO_PATH itself, not REPO_PATH/src, because src/ is a package.)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f'✓ Repo at: {REPO_PATH}')
print(f'✓ Project root on Python path')

Cloning into '/content/cv-face-alignment'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 37 (delta 11), reused 17 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 1.36 MiB | 4.46 MiB/s, done.
Resolving deltas: 100% (11/11), done.
✓ Repo at: /content/cv-face-alignment
✓ Project root on Python path


In [ ]:
# === Cell 4: Verify src/ modules import cleanly ===
from src import config, data, evaluate, io, features, models, visualise
print('✓ All modules import cleanly')
print(f'  Image size: {config.IMAGE_SIZE}')
print(f'  Landmarks per face: {config.N_LANDMARKS}')
print(f'  Flip indices: {config.FLIP_INDICES}')

✓ All modules import cleanly
  Image size: 256
  Landmarks per face: 5
  Flip indices: (1, 0, 2, 4, 3)


In [ ]:
# === Verify dataset files are in Drive ===
import os
data_dir = f'{DRIVE_PROJECT_DIR}/data'
print(f'Files in {data_dir}:')
for f in sorted(os.listdir(data_dir)):
    size_mb = os.path.getsize(f'{data_dir}/{f}') / (1024 * 1024)
    print(f'  {f}  ({size_mb:.1f} MB)')

Files in /content/drive/MyDrive/cv-face-alignment/data:
  face_alignment_test_images.npz  (74.8 MB)
  face_alignment_training_images.npz  (372.1 MB)
  train_val_split.npz  (0.0 MB)


In [ ]:
# === Inspect training data shape ===
import numpy as np

train_path = f'{DRIVE_PROJECT_DIR}/data/face_alignment_training_images.npz'
with np.load(train_path, allow_pickle=True) as data:
    print('Keys in the .npz:', list(data.keys()))
    for key in data.keys():
        arr = data[key]
        print(f'  {key}: shape={arr.shape}, dtype={arr.dtype}')

test_path = f'{DRIVE_PROJECT_DIR}/data/face_alignment_test_images.npz'
with np.load(test_path, allow_pickle=True) as data:
    print('\nTest keys:', list(data.keys()))
    for key in data.keys():
        arr = data[key]
        print(f'  {key}: shape={arr.shape}, dtype={arr.dtype}')

Keys in the .npz: ['images', 'points']
  images: shape=(2811, 256, 256, 3), dtype=uint8
  points: shape=(2811, 5, 2), dtype=float64

Test keys: ['images']
  images: shape=(554, 256, 256, 3), dtype=uint8
